Phase 18 — Production Hardening & Final API Response

Objective

Convert the successful internal ask_copilot() response into a clean, stable production response.

The key improvement is this:

Current RAG output

"answer": { "success": true, "answer": "The discount policy..." }

Target production output

"answer": "The discount policy..."

We will also preserve sources correctly.

In [0]:
# ================================================================
# PHASE 18 — PRODUCTION HARDENING & FINAL API RESPONSE
# ================================================================

print("=" * 70)
print("PHASE 18 — PRODUCTION HARDENING & FINAL API RESPONSE")
print("=" * 70)

print()
print("Purpose:")
print("- Validate production dependencies")
print("- Normalize SQL / RAG / Hybrid responses")
print("- Remove nested RAG response objects")
print("- Deduplicate sources")
print("- Convert Spark DataFrames into API-safe records")
print("- Produce a clean final API response")

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py

In [0]:
# ================================================================
# IMPORTS
# ================================================================

import json
import time

print("Imports successful.")

In [0]:
# ================================================================
# DEPENDENCY CHECK
# ================================================================

print("=" * 70)
print("DEPENDENCY CHECK")
print("=" * 70)

required_functions = [
    "classify_question",
    "determine_route",
    "run_sql_route",
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context",
    "generate_rag_answer",
    "run_hybrid_route",
    "assemble_hybrid_answer",
    "ask_copilot"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:
        failed_dependencies.append(
            function_name
        )

print()
print("Total dependencies:", len(required_functions))
print("Failed dependencies:", len(failed_dependencies))

if failed_dependencies:

    raise RuntimeError(
        "Missing production dependencies: "
        + ", ".join(failed_dependencies)
    )

print()
print("Dependency check: PASS")

Response Normalization

In [0]:
# ================================================================
# RAG ANSWER NORMALIZER
# ================================================================

def normalize_rag_answer(rag_result):
    """
    Convert the internal RAG response into clean answer text.
    """

    if rag_result is None:

        return None

    # ------------------------------------------------------------
    # RAG result returned as dictionary
    # ------------------------------------------------------------

    if isinstance(rag_result, dict):

        answer = rag_result.get(
            "answer"
        )

        if answer is None:
            return None

        return str(answer).strip()

    # ------------------------------------------------------------
    # RAG result already returned as text
    # ------------------------------------------------------------

    return str(
        rag_result
    ).strip()


print(
    "normalize_rag_answer(): PASS"
)

In [0]:
# ================================================================
# SOURCE NORMALIZER
# ================================================================

def normalize_sources(sources):
    """
    Remove duplicate sources while preserving order.
    """

    if not sources:

        return []

    normalized = []

    seen = set()

    for source in sources:

        if not isinstance(source, dict):
            continue

        source_key = (
            source.get("chunk_id"),
            source.get("document_id"),
            source.get("file_name"),
            source.get("title")
        )

        if source_key in seen:
            continue

        seen.add(
            source_key
        )

        normalized.append(
            {
                "chunk_id": source.get(
                    "chunk_id"
                ),
                "document_id": source.get(
                    "document_id"
                ),
                "file_name": source.get(
                    "file_name"
                ),
                "title": source.get(
                    "title"
                )
            }
        )

    return normalized


print(
    "normalize_sources(): PASS"
)

In [0]:
# ================================================================
# DATAFRAME NORMALIZER
# ================================================================

def normalize_sql_data(data):
    """
    Convert Spark DataFrame into JSON/API-safe records.
    """

    if data is None:

        return None

    # ------------------------------------------------------------
    # Spark DataFrame
    # ------------------------------------------------------------

    if hasattr(data, "collect"):

        rows = data.collect()

        return [
            row.asDict(
                recursive=True
            )
            for row in rows
        ]

    # ------------------------------------------------------------
    # Already a list
    # ------------------------------------------------------------

    if isinstance(data, list):

        normalized = []

        for item in data:

            if hasattr(item, "asDict"):

                normalized.append(
                    item.asDict(
                        recursive=True
                    )
                )

            elif isinstance(item, dict):

                normalized.append(
                    item
                )

            else:

                normalized.append(
                    str(item)
                )

        return normalized

    # ------------------------------------------------------------
    # Dictionary
    # ------------------------------------------------------------

    if isinstance(data, dict):

        return data

    # ------------------------------------------------------------
    # Fallback
    # ------------------------------------------------------------

    return str(data)


print(
    "normalize_sql_data(): PASS"
)

In [0]:
# ================================================================
# FINAL API RESPONSE FORMATTER
# ================================================================

def format_production_response(response):
    """
    Convert internal copilot response into a clean,
    consistent, API-safe response.
    """

    if response is None:

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Copilot returned None.",
            "execution_time_ms": None
        }

    if not isinstance(response, dict):

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                "Invalid response type: "
                + type(response).__name__
            ),
            "execution_time_ms": None
        }

    route = response.get(
        "route"
    )

    raw_answer = response.get(
        "answer"
    )

    # ============================================================
    # SQL RESPONSE
    # ============================================================

    if route == "sql":

        answer = (
            "SQL analysis completed successfully."
            if response.get("success")
            else None
        )

    # ============================================================
    # RAG RESPONSE
    # ============================================================

    elif route == "rag":

        answer = normalize_rag_answer(
            raw_answer
        )

    # ============================================================
    # HYBRID RESPONSE
    # ============================================================

    elif route == "hybrid":

        answer = raw_answer

        if isinstance(
            answer,
            dict
        ):

            answer = dict(
                answer
            )

            if "discount_policy" in answer:

                answer["discount_policy"] = (
                    normalize_rag_answer(
                        answer["discount_policy"]
                    )
                )

    # ============================================================
    # UNSUPPORTED
    # ============================================================

    else:

        answer = raw_answer

    return {
        "success": response.get(
            "success",
            False
        ),

        "question": response.get(
            "question"
        ),

        "route": route,

        "answer": answer,

        "sql": response.get(
            "sql"
        ),

        "data": normalize_sql_data(
            response.get("data")
        ),

        "sources": normalize_sources(
            response.get(
                "sources",
                []
            )
        ),

        "error": response.get(
            "error"
        ),

        "execution_time_ms": response.get(
            "execution_time_ms"
        )
    }


print(
    "format_production_response(): PASS"
)

In [0]:
# ================================================================
# HYBRID SOURCE EXTRACTION
# ================================================================

def extract_hybrid_sources(response):
    """
    Extract source information from the hybrid RAG result
    when available.
    """

    if not isinstance(
        response,
        dict
    ):
        return []

    if response.get("route") != "hybrid":
        return []

    hybrid_answer = response.get(
        "answer"
    )

    if not isinstance(
        hybrid_answer,
        dict
    ):
        return []

    discount_policy = hybrid_answer.get(
        "discount_policy"
    )

    if not isinstance(
        discount_policy,
        dict
    ):
        return []

    return normalize_sources(
        discount_policy.get(
            "sources",
            []
        )
    )


print(
    "extract_hybrid_sources(): PASS"
)

In [0]:
# ================================================================
# PRODUCTION FORMATTER — FINAL VERSION
# ================================================================

def build_final_api_response(response):
    """
    Build the final production API response.
    """

    formatted = format_production_response(
        response
    )

    # ------------------------------------------------------------
    # Add hybrid sources if available
    # ------------------------------------------------------------

    if formatted["route"] == "hybrid":

        hybrid_sources = extract_hybrid_sources(
            response
        )

        formatted["sources"] = normalize_sources(
            formatted["sources"]
            + hybrid_sources
        )

    # ------------------------------------------------------------
    # Final source normalization
    # ------------------------------------------------------------

    formatted["sources"] = normalize_sources(
        formatted["sources"]
    )

    return formatted


print(
    "build_final_api_response(): PASS"
)

In [0]:
# ================================================================
# TEST 1 — SQL
# ================================================================

print("=" * 70)
print("TEST 1 — SQL PRODUCTION RESPONSE")
print("=" * 70)

sql_question = (
    "Which region generated the highest revenue?"
)

sql_raw = ask_copilot(
    sql_question
)

sql_api = build_final_api_response(
    sql_raw
)

print(
    json.dumps(
        sql_api,
        indent=2,
        default=str
    )
)

assert sql_api["success"] is True
assert sql_api["route"] == "sql"
assert sql_api["error"] is None

print()
print("SQL PRODUCTION RESPONSE: PASS")

In [0]:
# ================================================================
# TEST 2 — RAG
# ================================================================

print("=" * 70)
print("TEST 2 — RAG PRODUCTION RESPONSE")
print("=" * 70)

rag_question = (
    "What is the discount policy?"
)

rag_raw = ask_copilot(
    rag_question
)

rag_api = build_final_api_response(
    rag_raw
)

print(
    json.dumps(
        rag_api,
        indent=2,
        default=str
    )
)

assert rag_api["success"] is True
assert rag_api["route"] == "rag"
assert rag_api["error"] is None

assert isinstance(
    rag_api["answer"],
    str
)

assert (
    "dummy answer"
    not in rag_api["answer"].lower()
)

print()
print("RAG PRODUCTION RESPONSE: PASS")

In [0]:
# ================================================================
# TEST 3 — HYBRID
# ================================================================

print("=" * 70)
print("TEST 3 — HYBRID PRODUCTION RESPONSE")
print("=" * 70)

hybrid_question = (
    "Which region generated the highest revenue "
    "and what discount policy applies there?"
)

hybrid_raw = ask_copilot(
    hybrid_question
)

hybrid_api = build_final_api_response(
    hybrid_raw
)

print(
    json.dumps(
        hybrid_api,
        indent=2,
        default=str
    )
)

assert hybrid_api["success"] is True
assert hybrid_api["route"] == "hybrid"
assert hybrid_api["error"] is None

assert isinstance(
    hybrid_api["answer"],
    dict
)

assert (
    hybrid_api["answer"].get(
        "highest_revenue_region"
    )
    is not None
)

assert (
    hybrid_api["answer"].get(
        "highest_revenue"
    )
    is not None
)

assert isinstance(
    hybrid_api["answer"].get(
        "discount_policy"
    ),
    str
)

print()
print("HYBRID PRODUCTION RESPONSE: PASS")

In [0]:
# ================================================================
# SOURCE DEDUPLICATION VALIDATION
# ================================================================

print("=" * 70)
print("SOURCE DEDUPLICATION VALIDATION")
print("=" * 70)

source_keys = []

for source in hybrid_api["sources"]:

    key = (
        source.get("chunk_id"),
        source.get("document_id"),
        source.get("file_name"),
        source.get("title")
    )

    source_keys.append(
        key
    )

unique_source_keys = set(
    source_keys
)

print(
    "Total sources:",
    len(source_keys)
)

print(
    "Unique sources:",
    len(unique_source_keys)
)

assert len(source_keys) == len(
    unique_source_keys
)

print()
print("Source deduplication: PASS")

In [0]:
# ================================================================
# JSON SERIALIZATION VALIDATION
# ================================================================

print("=" * 70)
print("JSON SERIALIZATION VALIDATION")
print("=" * 70)

production_responses = [
    sql_api,
    rag_api,
    hybrid_api
]

for response in production_responses:

    try:

        json_string = json.dumps(
            response,
            default=str
        )

        json.loads(
            json_string
        )

        print(
            f"PASS - {response['route'].upper()}"
        )

    except Exception as e:

        print(
            f"FAIL - {response.get('route')}"
        )

        raise e

print()
print("JSON serialization: PASS")

Final API Examples
Cell 16 — Clean RAG Output

In [0]:
# ================================================================
# FINAL RAG API RESPONSE
# ================================================================

print("=" * 70)
print("FINAL RAG API RESPONSE")
print("=" * 70)

print(
    json.dumps(
        rag_api,
        indent=2,
        ensure_ascii=False,
        default=str
    )
)

— Clean Hybrid Output

In [0]:
# ================================================================
# FINAL HYBRID API RESPONSE
# ================================================================

print("=" * 70)
print("FINAL HYBRID API RESPONSE")
print("=" * 70)

print(
    json.dumps(
        hybrid_api,
        indent=2,
        ensure_ascii=False,
        default=str
    )
)

Final Production Validation

In [0]:
# ================================================================
# PHASE 18 — FINAL PRODUCTION VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 18 — FINAL PRODUCTION VALIDATION")
print("=" * 70)

production_tests = [
    (
        "SQL",
        sql_api
    ),
    (
        "RAG",
        rag_api
    ),
    (
        "HYBRID",
        hybrid_api
    )
]

successful_tests = 0

for route_name, result in production_tests:

    passed = (
        result.get("success") is True
        and result.get("route") is not None
        and result.get("error") is None
        and result.get("answer") is not None
    )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{route_name}"
    )

    if passed:
        successful_tests += 1

print()
print(
    "Total production tests:",
    len(production_tests)
)

print(
    "Successful tests:",
    successful_tests
)

print(
    "Failed tests:",
    len(production_tests)
    - successful_tests
)

if successful_tests == len(
    production_tests
):

    print()
    print(
        "PHASE 18 STATUS: PASS ✓"
    )

else:

    print()
    print(
        "PHASE 18 STATUS: FAIL ✗"
    )

    raise RuntimeError(
        "Phase 18 production hardening failed."
    )

— Final Summary

In [0]:
# ================================================================
# PHASE 18 — FINAL SUMMARY
# ================================================================

print("=" * 70)
print("PHASE 18 — PRODUCTION HARDENING SUMMARY")
print("=" * 70)

summary = {
    "sql": sql_api["success"],
    "rag": rag_api["success"],
    "hybrid": hybrid_api["success"],
    "source_deduplication": (
        len(source_keys)
        == len(unique_source_keys)
    ),
    "json_serialization": True,
    "api_response_format": True
}

for check_name, passed in summary.items():

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{check_name}"
    )

print()
print("Production hardening complete.")
print("Final API response format: READY")